# 03 — Premier modèle CPU et RAM

Deux modèles de forêt aléatoire prédisent, chacun, la moyenne des consommations par tentative et le pic observé. Ils utilisent seulement `instance_num`, `plan_cpu` et `plan_mem`.

La comparaison est faite avec une prédiction constante égale à la moyenne des **données d'apprentissage**. Les paramètres de la forêt sont fixés avant l'évaluation : 100 arbres, profondeur maximale 12, au moins 5 observations par feuille.

La séparation 80/20 concerne les **jobs**, pas exactement le nombre de lignes. Elle est commune au CPU et à la RAM. Aucun réglage de paramètres ni choix du meilleur modèle n'utilise le jeu de test.

Ce notebook lit les résultats déjà calculés. Pour relancer depuis la racine du projet :
```powershell
.venv\Scripts\python.exe -m ML.src.train
```

In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd

project_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "ML" / "src" / "train.py").is_file()),
    None,
)
if project_root is None:
    raise RuntimeError("Ouvrir ce notebook depuis le projet ai-cloud-optimizer.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
model_dir = project_root / "ML" / "models" / "reference_v1"
report = json.loads((model_dir / "evaluation.json").read_text(encoding="utf-8"))
splits = pd.read_csv(model_dir / "job_split.csv")
pd.DataFrame(report["split"]).T

In [ ]:
assert not splits.job_id.duplicated().any()
train_jobs = set(splits.loc[splits.split.eq("train"), "job_id"])
test_jobs = set(splits.loc[splits.split.eq("test"), "job_id"])
assert not train_jobs & test_jobs
assert len(train_jobs) == report["split"]["train"]["jobs"]
assert len(test_jobs) == report["split"]["test"]["jobs"]
print("Vérification réussie : aucun job partagé entre apprentissage et test.")

## Erreurs sur les jobs de test

**MAE** : erreur absolue moyenne (plus bas = mieux). **RMSE** pénalise davantage les grosses erreurs. **R²** indique la variabilité expliquée sur le test ; il peut être négatif.

Le gain de MAE face à la constante **n'est pas un pourcentage de précision**. Les scores CPU et RAM ont des unités différentes : ne pas comparer directement leurs MAE.

In [ ]:
rows = []
for resource, details in report["resources"].items():
    for target, values in details["targets"].items():
        forest = values["forest_test"]
        rows.append({
            "target": target,
            "test_tasks": forest["rows"],
            "constant_MAE": values["baseline_test"]["mae"],
            "forest_MAE": forest["mae"],
            "MAE_reduction_percent": 100 * values["mae_reduction_vs_constant"] if values["mae_reduction_vs_constant"] is not None else None,
            "forest_RMSE": forest["rmse"],
            "forest_R2": forest["r2"],
            "forest_train_MAE": values["forest_train"]["mae"],
        })
metrics = pd.DataFrame(rows).set_index("target")
metrics

## Couverture des observations

Une couverture complète signifie que toutes les tentatives terminées **acceptées** possèdent des mesures valides. Cela ne garantit pas que toutes les instances demandées sont représentées.

Les groupes ci-dessous sont évalués séparément pour détecter les différences liées aux données manquantes.

In [ ]:
coverage_rows = []
for details in report["resources"].values():
    for target, values in details["targets"].items():
        for coverage, scores in values["test_by_coverage"].items():
            coverage_rows.append({"target": target, "coverage": coverage,
                "tasks": scores["forest"]["rows"], "forest_MAE": scores["forest"]["mae"],
                "constant_MAE": scores["baseline"]["mae"]})
pd.DataFrame(coverage_rows)

## Recalcul indépendant des scores

Les prédictions sur le test sont sauvegardées. On vérifie leur provenance et on recalcule la MAE sans appeler la fonction d'évaluation du script.

In [ ]:
for resource, details in report["resources"].items():
    predictions = pd.read_csv(model_dir / f"{resource}_test_predictions.csv")
    assert set(predictions.job_id) <= test_jobs
    assert not set(predictions.job_id) & train_jobs
    assert not predictions.duplicated(["job_id", "task_id"]).any()
    assert len(predictions) == details["test_rows"]
    for target, scores in details["targets"].items():
        recomputed_mae = (predictions[target] - predictions[f"forest_{target}"]).abs().mean()
        assert abs(recomputed_mae - scores["forest_test"]["mae"]) < 1e-10
print("Scores recalculés et vérifiés.")

## Exemple avec les modèles sauvegardés

L'exemple utilise des caractéristiques au format Alibaba. La fonction signale les valeurs situées hors des plages d'apprentissage ; rester dans ces plages ne garantit pas une prédiction fiable.

In [ ]:
from ML.src.predict import predict_resources
example = predict_resources(instance_num=2, plan_cpu=50, plan_mem=0.005, model_dir=model_dir)
example

## Interprétation et suite

Ces premiers modèles constituent un point de comparaison reproductible. Leurs erreurs restent importantes et les pics peuvent être sous-estimés. Ils ne sont pas intégrés à l'API de recommandation.

La RAM reste normalisée ; le CPU reste exprimé dans les valeurs d'origine des fichiers. Aucun résultat n'est une recommandation de vCPU ou de Go.

Une amélioration ultérieure devra choisir ses paramètres par validation sur les **jobs d'apprentissage**, sans se régler sur ces scores de test désormais connus. Une validation externe ou temporelle sera nécessaire pour démontrer une généralisation au-delà de cette trace. Le lien avec les entrées métier de l'API (`expected_users`, `workload_type`) reste à concevoir.

Références : [GroupShuffleSplit](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GroupShuffleSplit.html), [RandomForestRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html).